## Building Tiled Libraries for Flood Mapping

The process includes the following main steps:

* Download segment-based FLDPLN libraries 
* Clean up FSPs and segments and create FSP and segment info and library meta info files.
* Tile libraries
* Generate segment shapefile
* Assign stream order to segments
* Add stream order to FSPs and segments and create stream order network info file


## Import python packages

In [20]:
import sys
import time

# Set tools/scripts folder (YOU NEED TO CHANGE THIS)
fldplnToolFolder = r'C:\Users\jsong33\Desktop\Research\FIM\FLDPLN_FIM\Amite_h15_dh1_Resample\Tools'

# add the tools folder to sys.path to access the fldpln module
sys.path.append(fldplnToolFolder) 
# fldpln modules
from fldpln_reorg import *
from fldpln import *

## Download segment-based libraries

The original segment-based libraries are saved as individual folders. You can download the [Spring watershed/library](https://itprdkarsap.home.ku.edu/download/fldpln/KS_original_library_spring.zip) and unzip it. There are 3 types of files for the library:
* SLIE_segment*.mat, a MATLAB .mat file which store the FSP-FPP relations for a segment
* SLIE_KS_AOI_*_fsp_info.xlsx, an Excel file which stores FSP information
* segment_network_info_*.xlsx, an Excel file which stores segment information
where '\*' indicates the ID of the segment and '**' is the name of the library, in this case "spring".   

You can download [all Kansas libraries](https://itprdkarsap.home.ku.edu/download/fldpln/KS_original_library.zip) which includes the following libraries:
[ 'bigblue','chik_saltfork_medlodg','cotneo','ksrv','litark',
'lowark','lowrep','mdc','midkan','midsol',
'morv','nemaha','neosho','ninnescah','osagemarmaton',
'smoksalsol','spring','upark','upkan','uprep',
'upsal','upsmoky','upsol','verdigriscaney','walnut']

In [21]:
#ksLibNames = [ 'bigblue','chik_saltfork_medlodg','cotneo','ksrv','litark',
#                'lowark','lowrep','mdc','midkan','midsol',
#                'morv','nemaha','neosho','ninnescah','osagemarmaton',
#                'smoksalsol','spring','upark','upkan','uprep',
#                'upsal','upsmoky','upsol','verdigriscaney','walnut']

### Library issues?

I looked David's Verdigris library and see several issues:
* The library_*.mat files need to be renamed using the following command: ren library_segment*.mat SLIE_segment*.mat
* The verdigris_fsp_info.xlsx Excel file needs to renamed to SLIE_KS_AOI_verdigris_fsp_info.xlsx
* Don't need the fsp_info.mat file

### Download libraries from notebook

You can also download and unzip the libraries by using the code below (not implemented yet!). The main issue is that there is NOT a http-glob function available to filter the zip file in a library folder as their names may be different in different libraries!

In [22]:
#segUrl = r'https://kars.ku.edu/media/downloads/Kastens/KS_flood_library'
#segLocFolder = r'E:\FLDPLN\KS_original_library'
segLocFolder = r'C:\Users\jsong33\Desktop\Research\FIM\FLDPLN_FIM\Amite_h15_dh1_Resample\ARB_original_library'
allLibNames = ['Amite']

libs = allLibNames

#for lib in libs:
#    DownloadSegmentLibrary(segUrl,lib,segLocFolder)

## Prepare library spatial reference

Find out the cell size and spatial reference of the FLDPLN segment-based libraries. Save the spatial reference as an ESRI Shapefile's .prj file under the folder defined by varibale segLibFolder. All the libraries under the folder are assumed with the same spatial reference.

In [23]:
# # inputs for Spring library
# segLibFolder = r'D:\xingong\SpringWatershed\original_segment_library' # r'E:\FLDPLN\KS_original_library'
# cellSize = 5
# prjFileName = 'SpatialReference.prj' # library's spatial reference saved in shapefile's .prj file

# inputs
segLibFolder = r'C:\Users\jsong33\Desktop\Research\FIM\FLDPLN_FIM\Amite_h15_dh1_Resample\ARB_original_library' # r'E:\FLDPLN\KS_original_library'
cellSize = 10   
prjFileName = 'SpatialReference.prj' # library's spatial reference saved in shapefile's .prj file

## Create FSP and segment info and library metadata info files

Here, function CreateFspSegmentInfoSpatialReferenceFiles() prepares FSPs and segments for tiling and mapping. It involves the following main tasks:
* Clean up segments 
    * It removes segments from the segment table if they are not in the FSP table. 
    * If a removed segment is the downstream segment of another segment in the segment table, the upstream segment ID is set to 0 (i.e., watershed outlet). 
    * Those removed segments are usually close to or in waterbodies. By removing those segments, a library may have several separate watersheds/outlets! For example, neosho has 3 separate watersheds (segment 13, 104, 186 as the outlet segments). 
* Assign each FSP a unique IDs for identifying the FSPs with their IDs instead of their coordinates
* Calculate FSP and segment downstream distance for using in interpolating FSP depth of water from gauges

At the end, this step saves the FSPs and segments in two text files (fsps_info.csv and segments_info.csv) and library cell size and spatial reference in a JSON file (CellSizeSpatialReference.json) under the folder defined by variable tiledLibFolder.


In [24]:
# tiled library folder
# tiledLibFolder = r'D:\xingong\SpringWatershed\tiled10km_snz_library' #r'E:\FLDPLN\KS_tiled10km_snz_library'
tiledLibFolder = r'C:\Users\jsong33\Desktop\Research\FIM\FLDPLN_FIM\Amite_h15_dh1_Resample\ARB_tiled10km_snz_library' 

# libraries to be tiled. 
# Note that the tiled libraries will have that same name as the segment-based libraries except they are located under the tiledLibFolder!
# libNames = ['spring']
libNames = ['Amite']

# Create FSP and segment info files and library meta file (i.e., cell size and spatial reference)
for libName in libNames:
    print(f'Create FSP and segment info and library meta files for library: {libName} ...')
    CreateFspSegmentInfoSpatialReferenceFiles(segLibFolder,libName,cellSize,prjFileName,tiledLibFolder,libName)

Create FSP and segment info and library meta files for library: Amite ...


## Tile libraries

Make sure to go through the previous steps and tiled libraries already have the FSP and segment info files (i.e., fsp_info.csv and segment_info.csv) and metadat file (CellSizeSpatialReference.json) under their folders. This is the most time consuming step and may take hours for large libraries. 

In [25]:
# libraries to be tiled
libNames = ['Amite']

# define tile size and format
tileSize = 10000
tileFileFormat = 'snappy' # 'snappy' or 'mat'

# tile libraries
for libName in libNames: 
    print(f'Tile library: {libName} ...')
    TileLibrary(segLibFolder,libName,tiledLibFolder,libName,tileSize,tileFileFormat) 

Tile library: Amite ...
Calculate library extent ...
Library external border extent (minX, maxX, minY, maxY) : (669699.08423142, 736539.08423142, 3340590.17452762, 3466350.17452762)
Total number of FSP-FPP relations: 487073004
Number of (possible) tiles: 91
Tile extents:
 [(669699.08423142, 679699.08423142, 3340590.17452762, 3350590.17452762), (669699.08423142, 679699.08423142, 3350590.17452762, 3360590.17452762), (669699.08423142, 679699.08423142, 3360590.17452762, 3370590.17452762), (669699.08423142, 679699.08423142, 3370590.17452762, 3380590.17452762), (669699.08423142, 679699.08423142, 3380590.17452762, 3390590.17452762), (669699.08423142, 679699.08423142, 3390590.17452762, 3400590.17452762), (669699.08423142, 679699.08423142, 3400590.17452762, 3410590.17452762), (669699.08423142, 679699.08423142, 3410590.17452762, 3420590.17452762), (669699.08423142, 679699.08423142, 3420590.17452762, 3430590.17452762), (669699.08423142, 679699.08423142, 3430590.17452762, 3440590.17452762), (66969

Saving FSP-FPP relations in a file...
Number of unique FSPs in the tile: 42
Tile FSP extent (fspMinX,fspMaxX,fspMinY,fspMaxY):  (683599.08423142, 710609.08423142, 3355650.17452762, 3408850.17452762)
Tile FPP extent (fppMinX,fppMaxX,fppMinY,fppMaxY):  (669879.08423142, 679699.08423142, 3360590.17452762, 3370590.17452762)
Processing tile:  4
Tile extent (minX, maxX, minY, maxY) : (669699.08423142, 679699.08423142, 3370590.17452762, 3380590.17452762)
Number of segments interseting with the tile:  40
Total number of FSP-FPP relations in the tile: 2055724
Saving FSP-FPP relations in a file...
Number of unique FSPs in the tile: 37
Tile FSP extent (fspMinX,fspMaxX,fspMinY,fspMaxY):  (682189.08423142, 710609.08423142, 3355650.17452762, 3397190.17452762)
Tile FPP extent (fppMinX,fppMaxX,fppMinY,fppMaxY):  (675929.08423142, 679699.08423142, 3370590.17452762, 3380590.17452762)
Processing tile:  5
Tile extent (minX, maxX, minY, maxY) : (669699.08423142, 679699.08423142, 3380590.17452762, 3390590.1

Total number of FSP-FPP relations in the tile: 591108
Saving FSP-FPP relations in a file...
Number of unique FSPs in the tile: 8
Tile FSP extent (fspMinX,fspMaxX,fspMinY,fspMaxY):  (685249.08423142, 688149.08423142, 3411360.17452762, 3420180.17452762)
Tile FPP extent (fppMinX,fppMaxX,fppMinY,fppMaxY):  (681799.08423142, 688329.08423142, 3420590.17452762, 3430430.17452762)
Processing tile:  16
Tile extent (minX, maxX, minY, maxY) : (679699.08423142, 689699.08423142, 3430590.17452762, 3440590.17452762)
Number of segments interseting with the tile:  0
Total number of FSP-FPP relations in the tile: 0
Processing tile:  16
Tile extent (minX, maxX, minY, maxY) : (679699.08423142, 689699.08423142, 3440590.17452762, 3450590.17452762)
Number of segments interseting with the tile:  0
Total number of FSP-FPP relations in the tile: 0
Processing tile:  16
Tile extent (minX, maxX, minY, maxY) : (679699.08423142, 689699.08423142, 3450590.17452762, 3460590.17452762)
Number of segments interseting with 

Total number of FSP-FPP relations in the tile: 0
Processing tile:  28
Tile extent (minX, maxX, minY, maxY) : (699699.08423142, 709699.08423142, 3350590.17452762, 3360590.17452762)
Number of segments interseting with the tile:  45
Total number of FSP-FPP relations in the tile: 27769861
Saving FSP-FPP relations in a file...
Number of unique FSPs in the tile: 2292
Tile FSP extent (fspMinX,fspMaxX,fspMinY,fspMaxY):  (682579.08423142, 710609.08423142, 3351330.17452762, 3417040.17452762)
Tile FPP extent (fppMinX,fppMaxX,fppMinY,fppMaxY):  (699699.08423142, 709699.08423142, 3351400.17452762, 3360590.17452762)
Processing tile:  29
Tile extent (minX, maxX, minY, maxY) : (699699.08423142, 709699.08423142, 3360590.17452762, 3370590.17452762)
Number of segments interseting with the tile:  45
Total number of FSP-FPP relations in the tile: 31090443
Saving FSP-FPP relations in a file...
Number of unique FSPs in the tile: 241
Tile FSP extent (fspMinX,fspMaxX,fspMinY,fspMaxY):  (682579.08423142, 711249

Processing tile:  43
Tile extent (minX, maxX, minY, maxY) : (709699.08423142, 719699.08423142, 3370590.17452762, 3380590.17452762)
Number of segments interseting with the tile:  47
Total number of FSP-FPP relations in the tile: 14931647
Saving FSP-FPP relations in a file...
Number of unique FSPs in the tile: 42
Tile FSP extent (fspMinX,fspMaxX,fspMinY,fspMaxY):  (685169.08423142, 711249.08423142, 3355650.17452762, 3408850.17452762)
Tile FPP extent (fppMinX,fppMaxX,fppMinY,fppMaxY):  (709699.08423142, 717139.08423142, 3370590.17452762, 3380590.17452762)
Processing tile:  44
Tile extent (minX, maxX, minY, maxY) : (709699.08423142, 719699.08423142, 3380590.17452762, 3390590.17452762)
Number of segments interseting with the tile:  50
Total number of FSP-FPP relations in the tile: 2198965
Saving FSP-FPP relations in a file...
Number of unique FSPs in the tile: 15
Tile FSP extent (fspMinX,fspMaxX,fspMinY,fspMaxY):  (697349.08423142, 711249.08423142, 3355650.17452762, 3408850.17452762)
Tile F

Total number of FSP-FPP relations in the tile: 0
Processing tile:  55
Tile extent (minX, maxX, minY, maxY) : (719699.08423142, 729699.08423142, 3430590.17452762, 3440590.17452762)
Number of segments interseting with the tile:  10
Total number of FSP-FPP relations in the tile: 0
Processing tile:  55
Tile extent (minX, maxX, minY, maxY) : (719699.08423142, 729699.08423142, 3440590.17452762, 3450590.17452762)
Number of segments interseting with the tile:  8
Total number of FSP-FPP relations in the tile: 595128
Saving FSP-FPP relations in a file...
Number of unique FSPs in the tile: 88
Tile FSP extent (fspMinX,fspMaxX,fspMinY,fspMaxY):  (713969.08423142, 720099.08423142, 3437610.17452762, 3453220.17452762)
Tile FPP extent (fppMinX,fppMaxX,fppMinY,fppMaxY):  (719699.08423142, 725699.08423142, 3443060.17452762, 3450590.17452762)
Processing tile:  56
Tile extent (minX, maxX, minY, maxY) : (719699.08423142, 729699.08423142, 3450590.17452762, 3460590.17452762)
Number of segments interseting wit

## Generate segment shapefile

Create a shapefile with all the segments in all the libraries. This shapefile is used to assign each segment a stream order which is used in interpolating FSP depth of flow (DOF). 

In [26]:
# shpFolder = r'E:\FLDPLN\stream_order'
# shpName = "library_segments.shp" 
shpFolder = r'C:\Users\jsong33\Desktop\Research\FIM\FLDPLN_FIM\Amite_h15_dh1_Resample\Stream_order'
shpName = "library_segments.shp"

# libs included
libNames = ['Amite']

# generate segment shapefile 
print(f'Generate segment shapefile for libraries ...')
GenerateSegmentShapefiles(tiledLibFolder,libNames,shpFolder,shpName)

Generate segment shapefile for libraries ...


,geometry,LibName,SegId,StrOrd
0,"LINESTRING (704204.084 3457615.175, 704194.084...",Amite,1.0,NaN
1,"LINESTRING (704214.084 3456825.175, 704224.084...",Amite,2.0,NaN
2,"LINESTRING (705164.084 3453315.175, 705154.084...",Amite,3.0,NaN
3,"LINESTRING (705394.084 3451305.175, 705394.084...",Amite,4.0,NaN
4,"LINESTRING (705124.084 3447955.175, 705114.084...",Amite,5.0,NaN
...,...,...,...,...
75,"LINESTRING (710604.084 3360705.175, 710604.084...",Amite,76.0,NaN
76,"LINESTRING (710504.084 3360305.175, 710504.084...",Amite,77.0,NaN
77,"LINESTRING (702614.084 3358995.175, 702624.084...",Amite,78.0,NaN
78,"LINESTRING (706214.084 3357905.175, 706224.084...",Amite,79.0,NaN


## Assign segment stream order

Currently this is performed in GIS manually. The general rules for assigning stream order are:
* Low stream (multiple segments) order indicates main stream and high orders mean tributaries.
* Order must be unique to each contiguous stream. No two streams can have the same order even if the streams are in different sub-watersheds in a library (see Neosho as an example)
* Those are necessary conditions to make sure the interpolation on DOF can be carried out correctly.

The level path approach used in NWC may be used here to automate stream order assignment!

In [27]:
# code to automate, at least to some extent, the process

## Get stream order for FSPs and segments

This step gets stream orders for FSPs and segments from the stream order shapefile and save the information in the two text files. 

It also creates a new table, stream_order_info.csv, which stores the connectivity among stream orders with columns: [‘StrOrd’, ‘DsStrOrd’, ‘JunctionFspX’, ‘JunctionFspY’]. This information is used in DOF interpolation.

In [29]:
# lib included
libNames = ['Amite']

for libName in libNames:
    print(f'Get stream order and generate stream order network info for library: {libName} ...')
    GetStreamOrdersForFspsSegments(tiledLibFolder,libName,shpFolder,shpName)

Get stream order and generate stream order network info for library: Amite ...


## Distribute tiled libraries

Now we can make tiled libraries available for flood mapping. This involves 3 parts:
* Zip all the files for each tiled library as a single zip file. The code below can be used to automate this step
* The AHPS gauge file 
* Mapping scripts and notebooks

All the files are distributed at this FTP site.

In [30]:
# library folder
libFolder = r'C:\Users\jsong33\Desktop\Research\FIM\FLDPLN_FIM\Amite_h15_dh1_Resample\ARB_tiled10km_snz_library'

allLibNames = [ 'Amite']
# libraries to zip
libs = allLibNames
for libName in libs:
    print(f'Zipping library: {libName} ...')
    shutil.make_archive(os.path.join(libFolder,libName), 'zip', os.path.join(libFolder,libName))

Zipping library: Amite ...
